# Deep Learning / NLP Text Classifier — Spam Detection (PyTorch)

**Task:** Build an NLP pipeline that classifies text messages as **Spam** or **Ham** (not spam).

**Pipeline:**
1. Generate/load a labeled text dataset
2. Tokenize & vectorize text with **TF-IDF**
3. Build a multi-layer **Neural Network** in PyTorch with **Dropout**, **BatchNorm**, and **early stopping**
4. Train & validate the model, plotting loss/accuracy convergence curves
5. Evaluate on a held-out test set
6. Run inference on brand-new, unseen sample inputs and report classification confidence

> **Note:** This dataset is generated programmatically from templates so the notebook is fully
> self-contained and reproducible with no external downloads required. You can swap in any real
> labeled text dataset (e.g. SMS Spam Collection, IMDB reviews) by replacing the data-loading cell —
> everything downstream (vectorizer, model, training loop, evaluation) stays the same.


## 1. Imports

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 2. Build the dataset

We generate 1,000 labeled messages (500 spam, 500 ham) from varied templates with randomized
fillers (prizes, amounts, links, days, times) so the text has realistic variety rather than
being trivially memorizable.


In [ ]:
spam_templates = [
    "WIN a free {prize} now! Call {phone} to claim your prize",
    "Congratulations! You have won ${amount}. Claim now at {link}",
    "URGENT: Your account will be suspended. Click {link} immediately",
    "Free entry to win a {prize}! Text WIN to {number} now",
    "You have been selected for a ${amount} cash reward, reply YES",
    "Limited time offer! Get {prize} for FREE, click {link} now",
    "Your loan of ${amount} has been approved, call {phone} now",
    "Congrats! Claim your free {prize} before it expires, visit {link}",
    "URGENT: Verify your bank account now or lose access, click {link}",
    "You have won a lottery worth ${amount}! Contact {phone} to collect",
]

ham_templates = [
    "Hey, are we still meeting for lunch tomorrow?",
    "Can you send me the report by {day}?",
    "Thanks for your help earlier, really appreciate it!",
    "Don\'t forget we have a team meeting on {day} at {time}",
    "I finished the assignment, will submit it tonight",
    "Let\'s catch up over coffee sometime this {day}",
    "The weather looks great this {day}, want to go for a walk?",
    "Please review the attached document before {day}",
    "Happy birthday! Hope you have a wonderful day",
    "Can you pick up groceries on your way home?",
    "I\'ll be a few minutes late to the {time} call",
    "Great job on the presentation today, well done!",
]

prizes = ["iPhone", "vacation package", "gift card", "laptop", "cash prize"]
phones = ["08001234567", "07123456789", "09001112223"]
links = ["bit.ly/claim123", "secure-verify.com", "winbig-now.net"]
amounts = ["500", "1000", "5000", "250"]
numbers = ["88888", "12345", "70070"]
days = ["Monday", "Wednesday", "Friday", "the weekend"]
times = ["3pm", "10am", "noon"]

def fill(template):
    return template.format(
        prize=random.choice(prizes), phone=random.choice(phones),
        link=random.choice(links), amount=random.choice(amounts),
        number=random.choice(numbers), day=random.choice(days), time=random.choice(times),
    )

N_PER_CLASS = 500
data = []
for _ in range(N_PER_CLASS):
    data.append((fill(random.choice(spam_templates)), 1))  # 1 = spam
for _ in range(N_PER_CLASS):
    data.append((fill(random.choice(ham_templates)), 0))   # 0 = ham

random.shuffle(data)
df = pd.DataFrame(data, columns=["text", "label"])
print(df.shape)
df.head()


## 3. Tokenize & vectorize text (TF-IDF)

In [ ]:
texts = df["text"].tolist()
labels = df["label"].tolist()

# Train / validation / test split (70 / 15 / 15), stratified to keep class balance
X_train_txt, X_temp_txt, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.30, random_state=SEED, stratify=labels
)
X_val_txt, X_test_txt, y_val, y_test = train_test_split(
    X_temp_txt, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), stop_words="english")
X_train = vectorizer.fit_transform(X_train_txt).toarray().astype(np.float32)
X_val = vectorizer.transform(X_val_txt).toarray().astype(np.float32)
X_test = vectorizer.transform(X_test_txt).toarray().astype(np.float32)

y_train = np.array(y_train, dtype=np.int64)
y_val = np.array(y_val, dtype=np.int64)
y_test = np.array(y_test, dtype=np.int64)

INPUT_DIM = X_train.shape[1]
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Vocabulary size (feature dim): {INPUT_DIM}")


## 4. PyTorch `Dataset` / `DataLoader`

In [ ]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 32

train_ds = TextDataset(X_train, y_train)
val_ds = TextDataset(X_val, y_val)
test_ds = TextDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


## 5. Model — multi-layer neural network with Dropout & BatchNorm

A simple but complete feed-forward classifier:
`Input -> Linear -> BatchNorm -> ReLU -> Dropout` (x2) `-> Linear -> logits`


In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims=(256, 64), num_classes=2, dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            prev_dim = h
        layers.append(nn.Linear(prev_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = TextClassifier(INPUT_DIM).to(device)
print(model)


## 6. Training loop with early stopping

We track training/validation loss and accuracy every epoch, and stop early if validation loss
doesn't improve for `patience` consecutive epochs, restoring the best-performing weights.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

EPOCHS = 60
PATIENCE = 6

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

best_val_loss = float("inf")
epochs_no_improve = 0
best_state = None

def run_epoch(loader, train_mode):
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train_mode):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            if train_mode:
                optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            if train_mode:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += xb.size(0)
    return total_loss / total, correct / total

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_epoch(val_loader, train_mode=False)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:2d}/{EPOCHS} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        epochs_no_improve = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch} "
                  f"(no val_loss improvement for {PATIENCE} epochs).")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    print("Restored best model weights (lowest validation loss).")


## 7. Plot training curves (loss & accuracy convergence)

In [ ]:
epochs_ran = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(epochs_ran, history["train_loss"], label="Train Loss", marker="o")
axes[0].plot(epochs_ran, history["val_loss"], label="Val Loss", marker="o")
axes[0].set_title("Loss Convergence")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, history["train_acc"], label="Train Accuracy", marker="o")
axes[1].plot(epochs_ran, history["val_acc"], label="Val Accuracy", marker="o")
axes[1].set_title("Accuracy Convergence")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=120)
plt.show()


## 8. Evaluate on the held-out test set

In [ ]:
model.eval()
all_preds, all_true, all_probs = [], [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(yb.numpy())
        all_probs.extend(probs.cpu().numpy())

test_acc = accuracy_score(all_true, all_preds)
print(f"Test Accuracy: {test_acc:.4f}\n")
print(classification_report(all_true, all_preds, target_names=["Ham", "Spam"]))

cm = confusion_matrix(all_true, all_preds)
print("Confusion Matrix:")
print(pd.DataFrame(cm, index=["True Ham", "True Spam"], columns=["Pred Ham", "Pred Spam"]))


## 9. Inference on unseen sample inputs

Finally, we run the trained model on brand-new sentences it has never seen (not from the
training templates) and print the predicted class with its confidence score.


In [ ]:
def predict(texts_list):
    X_new = vectorizer.transform(texts_list).toarray().astype(np.float32)
    X_new_t = torch.tensor(X_new, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(X_new_t)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    results = []
    for text, p in zip(texts_list, probs):
        pred_class = "Spam" if p[1] > p[0] else "Ham"
        confidence = float(max(p))
        results.append((text, pred_class, confidence))
    return results

sample_inputs = [
    "Congratulations, you\'ve been chosen to win a brand new car, click here now!!!",
    "Hi mom, just checking in - I\'ll call you tonight after dinner.",
    "URGENT!! Your bank account has been locked, verify immediately or lose access",
    "Can we reschedule our meeting to Thursday afternoon instead?",
    "You have won a FREE cruise vacation, reply now to claim your spot!",
    "Loved the movie last night, we should go again sometime.",
]

results = predict(sample_inputs)

header_text, header_pred, header_conf = "Text", "Prediction", "Confidence"
print(f"{header_text:<70} {header_pred:<10} {header_conf}")
print("-" * 95)
for text, pred, conf in results:
    display_text = (text[:65] + "...") if len(text) > 65 else text
    print(f"{display_text:<70} {pred:<10} {conf:.2%}")


## Notes / next steps

- **Swap in real data:** replace the dataset-generation cell with a `pandas.read_csv(...)` on a
  real labeled dataset (e.g. SMS Spam Collection, IMDB reviews for sentiment). Everything else
  (vectorizer → PyTorch `Dataset` → model → training loop → evaluation → inference) works unchanged.
- **Try embeddings instead of TF-IDF:** swap `TfidfVectorizer` for Hugging Face tokenizer +
  pretrained embeddings (e.g. `bert-base-uncased`) for stronger performance on harder tasks.
- **Tune the model:** adjust `hidden_dims`, `dropout`, `lr`, or add more layers if you see
  underfitting; increase `weight_decay` or dropout if you see overfitting (val loss rising while
  train loss falls).
- **Requirements:** `torch`, `scikit-learn`, `pandas`, `numpy`, `matplotlib`
  (`pip install torch scikit-learn pandas numpy matplotlib`).
